# Compute Groundwater Time Series with Uncertainty Propagation

*This notebook guides you through computing a groundwater anomaly time series by subtracting snow mass (SNODAS), surface water (CDEC), and soil moisture (NLDAS) from total terrestrial water storage (GRACE), while propagating uncertainties. Ideal for graduate students new to water‑balance calculations.*

---

## 1. Introduction

In this analysis, we use four observational datasets:

- **GRACE**: Total terrestrial water storage anomaly (km³)
- **SNODAS**: Snow water equivalent anomaly (km³)
- **NLDAS**: Soil moisture and surface runoff anomaly (km³)
- **CDEC**: Reservoir storage anomaly (km³)

We compute the **groundwater** anomaly as:

> groundwater = GRACE - (SNODAS + NLDAS + CDEC)

Uncertainties (standard deviations) are propagated assuming independence:

> σ_gw = sqrt(σ_grace^2 + σ_snow^2 + σ_soil^2 + σ_res^2)

Throughout, we remove long-term means (2004–2009) to focus on anomalies.

---

## 2. Setup

In [1]:
# Standard imports
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Set display options for clarity
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', None)

*This cell imports the core libraries and configures pandas for better table display.*

---

## 3. Helper Functions

### 3.1 Loading a Time Series

In [ ]:
def load_series(path, date_col='date', target_day_of_month=15):
    """
    Load a CSV, parse dates, align to mid-month, and rename columns to ['value','error'].
    """
    df = pd.read_csv(
        path,
        converters={date_col: lambda x: pd.to_datetime(x).replace(day=target_day_of_month)}
    )
    df = df.set_index(date_col)
    df = df.resample('MS').first()
    df.index = df.index + pd.DateOffset(days=target_day_of_month-1)
    if len(df.columns) != 2:
        raise ValueError(f"Expected 2 columns, got {df.columns.tolist()}")
    df = df.rename(columns={df.columns[0]: 'value', df.columns[1]: 'error'})
    return df[['value', 'error']]

*Parses dates to the 15th of each month and ensures consistent monthly timestamps.*

### 3.2 Checking for Duplicate Months

In [ ]:
def show_monthly_duplicates(df, name):
    """Prints any duplicate timestamps so you can inspect overlaps."""
    dups = df.index[df.index.duplicated(keep=False)]
    if not dups.empty:
        print(f"{name} duplicates on months:")
        display(df.loc[dups])

*Duplicates often arise if two files contain overlapping data for the same basin.*

### 3.3 Removing Long‑Term Mean

In [ ]:
def remove_mean(series, start, end):
    """
    Subtracts the mean of `series` between `start` and `end` from all values.
    """
    μ = series.loc[start:end].mean()
    return series - μ

*Center each time series around zero by removing its baseline over the calibration period.*

### 3.4 Computing Groundwater and Error

In [ ]:
def compute_groundwater(grace, snodas, nldas, cdec, start, end):
    # Align and merge datasets
    df = pd.DataFrame({
        'grace': grace['value'],
        'snow':  snodas['value'],
        'soil':  nldas['value'],
        'res':   cdec['value'],
        'err_grace':  grace['error'],
        'err_snow':   snodas['error'],
        'err_soil':   nldas['error'],
        'err_res':    cdec['error'],
    }).dropna()

    # Demean each component
    for comp in ['grace','snow','soil','res']:
        df[comp] = remove_mean(df[comp], start, end)

    # Compute anomaly and propagate uncertainties
    df['groundwater'] = df['grace'] - (df['snow'] + df['soil'] + df['res'])
    df['error'] = np.sqrt(
        df['err_grace']**2 + df['err_snow']**2 + df['err_soil']**2 + df['err_res']**2
    )
    return df[['groundwater', 'error']]

*This function returns a monthly groundwater anomaly series with propagated uncertainties.*

### 3.5 Smoothing and Averaging

In [ ]:
def smooth_timeseries(sw, window=3):
    """Centered moving average (months)."""
    return sw.rolling(window=window, center=True, min_periods=1).mean()

def average_timeseries(sw, year_type='calendar'):
    """
    Compute yearly means (calendar or water year).
    """
    if year_type=='calendar':
        yearly = sw.resample('YE-DEC').mean()
        yearly.index = yearly.index - pd.DateOffset(months=6)
    else:
        df = sw.copy(); df['wy'] = df.index.to_series().apply(lambda d: d.year+1 if d.month>=10 else d.year)
        yearly = df.groupby('wy')[['groundwater','error']].mean()
        yearly.index = pd.to_datetime(yearly.index.astype(str)+'-09-30') - pd.DateOffset(months=6)
    yearly.index.name = 'date'
    return yearly

*Smoothing clarifies trends; yearly averages help compare interannual variability.*

---

## 4. Data Loading

In [ ]:
# Define filepaths (update these for your local environment)
base = Path('data')  # or wherever you store your CSVs
paths = {
    'grace': base / 'grace.csv',
    'snodas': base / 'snodas.csv',
    'nldas': base / 'nldas.csv',
    'cdec': base / 'cdec.csv',
}

# Load each series
grace  = load_series(paths['grace'])
snodas = load_series(paths['snodas'])
nldas  = load_series(paths['nldas'])
cdec   = load_series(paths['cdec'])

# Inspect loaded data
print("GRACE sample:")
display(grace.head())

*Ensure your files are accessible and correctly formatted before proceeding.*

---

## 5. Groundwater Calculation

In [ ]:
# Calibration period for baseline removal
cal_start, cal_end = '2004-01-01', '2009-12-31'

# Compute groundwater anomaly
sw = compute_groundwater(grace, snodas, nldas, cdec, cal_start, cal_end)
print("Computed groundwater series:")
display(sw.head())


---
## 6. Smoothing and Yearly Averages

In [ ]:
# 3‑month centered smoothing
sw_smoothed = smooth_timeseries(sw, window=3)

# Calendar-year and water-year averages
sw_cal = average_timeseries(sw, 'calendar')
sw_wat = average_timeseries(sw, 'water')


---
## 7. Visualization

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(sw.index, sw['groundwater'], label='Raw', linewidth=1)
plt.plot(sw_smoothed.index, sw_smoothed['groundwater'], label='Smoothed', linewidth=2)
plt.xlabel('Date')
plt.ylabel('Groundwater anomaly (km³)')
plt.title('Monthly Groundwater Anomaly')
plt.legend();

*Visual checks help verify that the smoothing behaves as expected.*

---

## 8. Saving Results

In [ ]:
out_dir = Path('output'); out_dir.mkdir(exist_ok=True)

sw.to_csv(out_dir / 'groundwater_monthly_unsmoothed.csv')
sw_smoothed.to_csv(out_dir / 'groundwater_monthly_smoothed.csv')
sw_cal.to_csv(out_dir / 'groundwater_calendar_year.csv')
sw_wat.to_csv(out_dir / 'groundwater_water_year.csv')
print("All outputs saved to 'output/' directory.")